In [ ]:
import os

import matplotlib.pyplot as plt
import numpy as np
import polars as pl

import metapop as mp

In [ ]:
def plot_calculations(joined_df, suffix="", scenario=""):
    immunity_old, immunities_old = mp.calculate_baseline_immunity_from_dataframe(
        joined_df
    )
    immunity_new, immunities_new = (
        mp.calculate_baseline_immunity_from_dataframe_average(joined_df)
    )
    immunity_3, immunities_3 = mp.calculate_baseline_immunity_from_dataframe_linear(
        joined_df
    )
    fig, ax = plt.subplots(figsize=(8.5, 5))

    ax.scatter(
        joined_df["coverage_range_min_age"],
        joined_df["threshold_coverage"],
        marker="s",
        c="#ff5f5f",
        s=90,
        zorder=0,
        label="Threshold coverage data",
    )

    ax.plot(
        joined_df["coverage_range_min_age"],
        joined_df["threshold_coverage"],
        linestyle="--",
        c="#68ade6",
        label="Ideal interpolation in continuous mode",
    )

    for idx, row in enumerate(joined_df.iter_rows(named=True)):
        d = pl.DataFrame([row])
        ax.scatter(
            np.arange(
                d["coverage_range_min_age"][0], d["coverage_range_max_age"][0], 1
            ),
            [d["threshold_coverage_mid"][0]]
            * (d["coverage_range_max_age"][0] - d["coverage_range_min_age"][0]),
            marker="x",
            s=100,
            c="#2431c9",
            zorder=1,
            label=f"Old discretized immunity: {immunity_old * 100:.0f}%"
            if idx == 0
            else None,
        )

        # x = np.arange(
        #     d["coverage_range_min_age"][0], d["coverage_range_max_age"][0] + 1, 1
        # )
        # y = []
        # for i in x:
        #     if i == d["coverage_range_min_age"][0]:
        #         y.append(d["threshold_coverage"][0])
        #     elif i == d["coverage_range_max_age"][0]:
        #         y.append(d["threshold_coverage_upper"][0])
        #     elif i == 1:
        #         y.append(
        #             joined_df.filter(pl.col("coverage_range_min_age") == 0)[
        #                 "threshold_coverage_mid"
        #             ][0]
        #         )
        #     else:
        #         y.append(d["threshold_coverage_mid"][0])

        # ax.scatter(
        #     x,
        #     y,
        #     marker="o",
        #     c="#28a745",
        #     s=80,
        #     zorder=2,
        #     label=f"New discretized immunity: {immunity_new * 100:.0f}%"
        #     if idx == 0
        #     else None,
        # )
    ax.scatter(
        np.arange(len(immunities_new)),
        immunities_new,
        marker="o",
        # c="#ffa500",
        c="#28a745",
        zorder=3,
        label=f"2nd discretized immunity: {immunity_3 * 100:.0f}%",
    )

    ax.scatter(
        np.arange(len(immunities_3)),
        immunities_3,
        marker="D",
        c="#912DBF",
        s=90,
        zorder=4,
        label=f"3rd discretized immunity: {immunity_3 * 100:.0f}%",
    )

    ax.set_xlim(-1, 20)
    ax.set_ylim(-1, 105)
    ax.set_xticks(range(0, 21, 1))
    ax.set_yticks(range(0, 110, 10))
    ax.set_xlabel("Age (years)")
    ax.set_ylabel("Vaccine coverage (%)")
    ax.set_title(f"Baseline Immunity Calculation Sketch{scenario}")

    leg = ax.legend(loc="upper right")
    leg.draw_frame(True)

    fig.savefig(
        os.path.join(
            os.path.dirname(os.path.abspath("__file__")),
            f"baseline_immunity_sketch_plot{suffix}.png",
        ),
        dpi=300,
    )

In [ ]:
config_path = os.path.join(
    os.path.dirname(os.path.abspath("__file__")),
    "..",
    "..",
    "metapop",
    "app_assets",
    "one_pop_config.yaml",
)

parms = mp.read_parameters(config_path)

pop_table = mp.initialize_pop_table(parms)
vacc_table = mp.initialize_vacc_table(parms)
# get range mappings
pop_table = mp.add_pop_ranges_to_pop_table(pop_table)
vacc_table = mp.add_threshold_values_to_vacc_table(vacc_table)

In [ ]:
# scenario: all population under 5 years old age group, full coverage at 5 only
pop_table = pop_table.with_columns(
    pl.when(pl.col("population") == "<5").then(100.0).otherwise(0.0).alias("percentage")
)

vacc_table = vacc_table.with_columns(
    pl.when((pl.col("threshold") == "5 years (kindergarten)"))
    .then(100.0)
    .otherwise(0.0)
    .alias("coverage")
)
immunity_df = mp.build_initial_baseline_immunity_dataframe_from_user_inputs(
    pop_table,
    vacc_table,
)

print(
    "Scenario: all population under 5 years old age group, full coverage for all eligible age groups"
)
joined_df = mp.create_dataframe_for_baseline_immunity_calculation(immunity_df)
print(joined_df)

immunity, immunities = mp.calculate_baseline_immunity_from_dataframe(joined_df)
print(f"Final baseline immunity value: {immunity * 100:.0f}%")

immunity_2, immunities_2 = mp.calculate_baseline_immunity_from_dataframe_average(
    joined_df
)
print(f"Final baseline immunity value 2: {immunity_2 * 100:.0f}%\n")

plot_calculations(
    joined_df,
    suffix="_under_5_at_5_vaccinated",
    scenario=": (all under 5, at 5 fully vaccinated)",
)

In [ ]:
# scenario: all population 5 to 17 years old age group, full coverage under 5 only
pop_table = pop_table.with_columns(
    pl.when(pl.col("population") == "5-17")
    .then(100.0)
    .otherwise(0.0)
    .alias("percentage")
)

vacc_table = vacc_table.with_columns(
    pl.when(
        (pl.col("threshold") == "2 years")
        | (pl.col("threshold") == "5 years (kindergarten)")
    )
    .then(100.0)
    .otherwise(0.0)
    .alias("coverage")
)
immunity_df = mp.build_initial_baseline_immunity_dataframe_from_user_inputs(
    pop_table,
    vacc_table,
)

print(
    "Scenario: all population 5 to 17 years old age group, full coverage under 5 only"
)
joined_df = mp.create_dataframe_for_baseline_immunity_calculation(immunity_df)
print(joined_df)

immunity, immunities = mp.calculate_baseline_immunity_from_dataframe(joined_df)
print(f"Final baseline immunity value: {immunity * 100:.0f}%")

immunity_2, immunities_2 = mp.calculate_baseline_immunity_from_dataframe_average(
    joined_df
)
print(f"Final baseline immunity value 2: {immunity_2 * 100:.0f}%\n")

plot_calculations(
    joined_df,
    suffix="_5_to_17_under_5_vaccinated",
    scenario=": (all 5-17, under 5 fully vaccinated)",
)

In [ ]:
# scenario: all population 5 to 17 years old age group, full coverage at 5 only
pop_table = pop_table.with_columns(
    pl.when(pl.col("population") == "5-17")
    .then(100.0)
    .otherwise(0.0)
    .alias("percentage")
)

vacc_table = vacc_table.with_columns(
    pl.when((pl.col("threshold") == "5 years (kindergarten)"))
    .then(100.0)
    .otherwise(0.0)
    .alias("coverage")
)
immunity_df = mp.build_initial_baseline_immunity_dataframe_from_user_inputs(
    pop_table,
    vacc_table,
)

print("Scenario: all population 5 to 17 years old age group, full coverage at 5 only")
joined_df = mp.create_dataframe_for_baseline_immunity_calculation(immunity_df)
print(joined_df)

immunity, immunities = mp.calculate_baseline_immunity_from_dataframe(joined_df)
print(f"Final baseline immunity value: {immunity * 100:.0f}%")

immunity_2, immunities_2 = mp.calculate_baseline_immunity_from_dataframe_average(
    joined_df
)
print(f"Final baseline immunity value 2: {immunity_2 * 100:.0f}%\n")

plot_calculations(
    joined_df,
    suffix="_5_to_17_at_5_vaccinated",
    scenario=": (all 5-17, at 5 fully vaccinated)",
)

In [ ]:
# scenario: all population under 5 years old age group, full coverage at 2 only
pop_table = pop_table.with_columns(
    pl.when(pl.col("population") == "<5").then(100.0).otherwise(0.0).alias("percentage")
)

vacc_table = vacc_table.with_columns(
    pl.when((pl.col("threshold") == "2 years"))
    .then(100.0)
    .otherwise(0.0)
    .alias("coverage")
)
immunity_df = mp.build_initial_baseline_immunity_dataframe_from_user_inputs(
    pop_table,
    vacc_table,
)

print("Scenario: all population under 5 years old age group, full coverage at 2 only")
joined_df = mp.create_dataframe_for_baseline_immunity_calculation(immunity_df)
print(joined_df)

immunity, immunities = mp.calculate_baseline_immunity_from_dataframe(joined_df)
print(f"Final baseline immunity value: {immunity * 100:.0f}%")

immunity_2, immunities_2 = mp.calculate_baseline_immunity_from_dataframe_average(
    joined_df
)
print(f"Final baseline immunity value 2: {immunity_2 * 100:.0f}%\n")

plot_calculations(
    joined_df,
    suffix="_under_5_at_2_vaccinated",
    scenario=": (all under 5, at 2 fully vaccinated)",
)

In [ ]:
# scenario: all population 5 to 17 years old age group, full coverage at 13 only
pop_table = pop_table.with_columns(
    pl.when(pl.col("population") == "5-17")
    .then(100.0)
    .otherwise(0.0)
    .alias("percentage")
)

vacc_table = vacc_table.with_columns(
    pl.when((pl.col("threshold") == "13 years"))
    .then(100.0)
    .otherwise(0.0)
    .alias("coverage")
)
immunity_df = mp.build_initial_baseline_immunity_dataframe_from_user_inputs(
    pop_table,
    vacc_table,
)

print("Scenario: all population 5 to 17 years old age group, full coverage at 13 only")
joined_df = mp.create_dataframe_for_baseline_immunity_calculation(immunity_df)
print(joined_df)

immunity, immunities = mp.calculate_baseline_immunity_from_dataframe(joined_df)
print(f"Final baseline immunity value: {immunity * 100:.0f}%")

immunity_2, immunities_2 = mp.calculate_baseline_immunity_from_dataframe_average(
    joined_df
)
print(f"Final baseline immunity value 2: {immunity_2 * 100:.0f}%\n")

plot_calculations(
    joined_df,
    suffix="_5_to_17_at_13_vaccinated",
    scenario=": (all 5-17, at 13 fully vaccinated)",
)

In [ ]:
# scenario: all population 18 years old age group, full coverage at 13 only
pop_table = pop_table.with_columns(
    pl.when(pl.col("population") == "18+")
    .then(100.0)
    .otherwise(0.0)
    .alias("percentage")
)

vacc_table = vacc_table.with_columns(
    pl.when((pl.col("threshold") == "13 years"))
    .then(100.0)
    .otherwise(0.0)
    .alias("coverage")
)
immunity_df = mp.build_initial_baseline_immunity_dataframe_from_user_inputs(
    pop_table,
    vacc_table,
)

print("Scenario: all population 18 years old age group, full coverage at 13 only")
joined_df = mp.create_dataframe_for_baseline_immunity_calculation(immunity_df)
print(joined_df)

immunity, immunities = mp.calculate_baseline_immunity_from_dataframe(joined_df)
print(f"Final baseline immunity value: {immunity * 100:.0f}%")

immunity_2, immunities_2 = mp.calculate_baseline_immunity_from_dataframe_average(
    joined_df
)
print(f"Final baseline immunity value 2: {immunity_2 * 100:.0f}%\n")

plot_calculations(
    joined_df,
    suffix="_18_at_13_vaccinated",
    scenario=": (all 18+, at 13 fully vaccinated)",
)